In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from scipy.ndimage import gaussian_filter
from utils import *
from cavity_correction import correct_cavity
from prefilter_correction import correct_prefilter

In [2]:
flat_files = sorted(glob.glob('../process/temp/*flat*.fits'))
ghost_files = sorted(glob.glob('../process/temp/*ghost*.fits'))
cavity_files = sorted(glob.glob('../process/temp/*cavity*.fits'))

print(flat_files)

['../process/temp/phi-fdt-flat_20240330T050009_V202608181831C_0463300100.fits', '../process/temp/phi-fdt-flat_20240926T114503_V202608181846C_0469260100.fits', '../process/temp/phi-fdt-flat_20241016T113003_V202608181902C_0470160100.fits', '../process/temp/phi-fdt-flat_20241027T233003_V202608181915C_0470270100.fits', '../process/temp/phi-fdt-flat_20241202T123003_V202608181931C_0472020100.fits', '../process/temp/phi-fdt-flat_20250119T210009_V202608181947C_0561190100.fits', '../process/temp/phi-fdt-flat_20250310T080009_V202608182002C_0563100100.fits', '../process/temp/phi-fdt-flat_20250915T140003_V202608182017C_0569150100.fits', '../process/temp/phi-fdt-flat_20250923T000503_V202608182032C_0569230100.fits', '../process/temp/phi-fdt-flat_20260310T040003_V202608182047C_0663100100.fits', '../process/temp/phi-fdt-flat_20260425T230003_V202608182102C_0664250100.fits']


In [3]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt'

i = -5

cavity_file, flat_file, ghost_file = cavity_files[i], flat_files[i], ghost_files[i]

with fits.open(dark_file) as hdul:
    dark = hdul[0].data

with fits.open(cavity_file) as hdul:
    cavity = hdul[0].data

with fits.open(flat_file) as hdul:
    flat = hdul[0].data
    header = hdul[0].header

with fits.open(ghost_file) as hdul:
    ghost = hdul[0].data

#ghost = demodulate(ghost, header)
flat_ = demodulate(flat, header)
flat_[1:] /= flat_[0]
flat_ -= np.mean(flat_, axis=(-2,-1), keepdims=True)

In [4]:
plt.figure(figsize=(10,10))
plt.imshow(cavity, 'bwr', vmin=-5e-2, vmax=5e-2)
plt.tight_layout()

In [5]:
plt.figure(figsize=(10,10))
plt.imshow(flat_[3], 'gray', vmin=-5e-3, vmax=5e-3)
plt.tight_layout()

In [6]:
plt.figure(figsize=(10,10))
plt.imshow(flat[0], 'gray', vmin=0.8, vmax=1.1)
plt.tight_layout()

In [7]:
folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/'
#folder = '/home/ulyanov/data/solo/phi/test/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T080009_V202503131733C_0563100100.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T080608_V202503131733C_0563100125.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T081208_V202503131835C_0563100150.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T081808_V202503131935C_0563100175.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T082408_V202503131935C_0563100200.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T083008_V202503132033C_0563100225.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T083608_V202503141634C_0563100250.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025

In [8]:
with fits.open(files[1]) as hdul:
    header = hdul[0].header
    data = hdul[0].data

xr, yr = reflection_point_predict(header)
wv = read_wavelengths(header)
cpos = header['CONTPOS'] - 1
print(cpos)

nx, ny = data.shape[-2:]

data = data.reshape(6,4,nx,ny)
data -= crop(dark, header) * 0.4
data = correct_prefilter(data, header, prefilter_file)
data /= crop(flat, header)

#data = correct_cavity(data, header, cavity)
#data = realign(data)

5


In [9]:
wv_shift = get_wv_shift(data, header)

plt.figure(figsize=(10,10))
plt.imshow(wv_shift, 'bwr', vmin=-5e-2, vmax=5e-2)
plt.tight_layout()

In [32]:
sigma = 0.043
gamma = 0.053
contpos = cpos
acc = 1e-3
lam = 1e-6

n_wv = len(wv)

wv_min = np.min(np.delete(wv, contpos) if contpos is not None else wv)
wv_max = np.max(np.delete(wv, contpos) if contpos is not None else wv)
wvc = (wv_min + wv_max) / 2
wv_ = np.arange(wv_min, wv_max + acc / 2, acc, dtype=np.float32)
n_wv_ = len(wv_)

xi = np.expand_dims(wv, axis=1) - np.expand_dims(wv_, axis=0)
A = voigt_profile(xi, sigma, gamma)
A0 = np.mean(A, axis=0, keepdims=True)
A -= A0

M = np.zeros((n_wv, n_wv))
N = np.zeros((n_wv, n_wv))

for k in range(n_wv_):
    gk = voigt_profile(wv_[k] - wvc, sigma, gamma) ** 2

    for j in range(n_wv):
        mjk = gk * (voigt_profile(wv_[k] - wv[j], sigma, gamma, modified=True) * 2 - A0[0,k])
        njk = gk * A[j,k]

        for i in range(n_wv):
            M[j, i] += mjk * A[i,k]
            N[j, i] += njk * A[i,k]

Q = M @ np.linalg.inv(N + lam * np.identity(n_wv))
Q = Q @ (np.identity(n_wv) - 1 / n_wv) + 1 / n_wv

In [24]:
Q

array([[ 9.43999756e-02,  2.63655138e-01,  5.00683558e-02,
         6.93062291e-03, -4.43963820e-02,  6.29342289e-01],
       [ 3.53698680e-01, -1.36193288e-01,  3.51385522e-01,
         2.99470420e-03, -9.88163080e-03,  4.37996012e-01],
       [ 3.89077700e-02,  3.35257381e-01, -1.29756919e-01,
         3.41774840e-01, -1.47320908e-02,  4.28549018e-01],
       [ 4.30803112e-02, -9.65530278e-04,  3.48060849e-01,
        -1.24241686e-01,  2.92407883e-01,  4.41658173e-01],
       [ 3.20456116e-02,  7.56742602e-04,  4.58729786e-02,
         2.79971790e-01,  7.34902692e-03,  6.34003851e-01],
       [ 2.58519248e-02, -1.92269378e-02,  2.72938813e-02,
        -1.86194885e-02,  6.86735432e-02,  9.16027077e-01]])

In [33]:
reflection = data[:,0].copy()
reflection = gaussian_filter(reflection, 8, axes=(-2,-1))
reflection = reflect(reflection, xr, yr)

reflection = np.matmul(Q, reflection, axes=[(-2, -1), (0, 1), (0, 1)])

temp = data - np.expand_dims(reflection, 1) * np.expand_dims(crop(ghost, header), 0)
temp = demodulate(temp, header)

In [36]:
i = 1
j = 2

a, b = np.nanpercentile(temp[i,0], 0.1), np.nanpercentile(temp[i,0], 99.9)

plt.figure(figsize=(10,10))
plt.imshow(temp[i,j], 'gray', vmin=-1e-3 * (b - a), vmax=1e-3 * (b - a))#, origin='lower')
plt.tight_layout()

In [27]:
reflection = data[:,0].copy()
reflection = np.matmul(Q, reflection, axes=[(-2, -1), (0, 1), (0, 1)])

In [28]:
plt.figure(figsize=(10,10))
plt.imshow(reflection[3] / data[3,0], vmin=0.7, vmax=1.3)
plt.tight_layout()

/tmp/ipykernel_103556/3425980744.py:2: RuntimeWarning: divide by zero encountered in divide
  plt.imshow(reflection[3] / data[3,0], vmin=0.7, vmax=1.3)


In [29]:
temp_ = calc_continuum(temp, header)

In [30]:
plt.figure(figsize=(10,10))
plt.imshow(temp_[3], 'gray', vmin=-1e-3 * (b - a), vmax=1e-3 * (b - a))#, origin='lower')
plt.tight_layout()

In [57]:
plt.figure(figsize=(10,10))
plt.imshow(temp[cpos,3] - temp_[3], 'gray', vmin=-1e-3 * (b - a), vmax=1e-3 * (b - a))#, origin='lower')
plt.tight_layout()